# 01 — EDA & Data Validation

This notebook serves two purposes:
1. **Verify data quality** from the Phase 1.2 `DataLoader` before any strategy is built on top.
2. **Extract statistical insights** about return distributions, volatility regimes, and
   cross-asset correlation that will directly inform strategy design in Phases 3.1–3.3.

**Sections**
- Data quality & validation
- Price & volume overview
- Return distributions
- Correlation & rolling metrics

In [ ]:
%matplotlib inline
import sys
import warnings
import pathlib

_here = pathlib.Path(".").resolve()
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from backtester import DataLoader
from config import DEFAULT_TICKERS, BENCHMARK_TICKER, START_DATE, END_DATE

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 5)

print("Environment ready.")

In [ ]:
# Load the equity universe plus the benchmark so correlation / performance
# comparisons against SPY are available throughout the notebook.
all_tickers = list(DEFAULT_TICKERS) + [BENCHMARK_TICKER]

loader  = DataLoader(tickers=all_tickers, start_date=START_DATE, end_date=END_DATE)
df_long = loader.load()
report  = loader.validate(df_long)
df_wide = loader.load_wide()
returns = loader.get_returns()

n_tickers = df_long["ticker"].nunique()
n_rows    = len(df_long)
start     = df_long.index.min().date()
end       = df_long.index.max().date()
print(f"Loaded {n_tickers} tickers, {n_rows:,} rows, {start} to {end}")

if report["missing_values"]:
    print(f"Missing values: {report['missing_values']}")
if report["gaps"]:
    print(f"Gaps detected in: {list(report['gaps'].keys())}")
if report["stale_tickers"]:
    print(f"Stale tickers: {report['stale_tickers']}")

## Section 1 — Data quality

In [ ]:
# -- Completeness per ticker -----------------------------------------------
completeness   = df_wide.notna().sum() / len(df_wide) * 100
tickers_sorted = completeness.sort_values(ascending=False).index.tolist()
bar_colors     = ["steelblue" if v >= 95 else "red" for v in completeness[tickers_sorted]]

# -- Year tick positions for the heatmap y-axis ----------------------------
yearly_idx     = df_wide.resample("YE").first().index
year_labels    = yearly_idx.year.tolist()
year_positions = [
    df_wide.index.get_indexer([d], method="nearest")[0]
    for d in yearly_idx
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: missing data heatmap
sns.heatmap(
    df_wide[tickers_sorted].isnull(),
    ax=axes[0],
    cmap="Reds",
    cbar=False,
    linewidths=0.3,
)
axes[0].set_title("Missing close prices", fontweight="bold")
axes[0].set_xlabel("Ticker")
axes[0].set_yticks(year_positions)
axes[0].set_yticklabels(year_labels, rotation=0)

# Right: completeness bar chart
axes[1].bar(tickers_sorted, completeness[tickers_sorted], color=bar_colors)
axes[1].axhline(95, color="red", linestyle="--", linewidth=1.5, label="95% threshold")
axes[1].set_title("Data completeness per ticker (%)", fontweight="bold")
axes[1].set_xlabel("Ticker")
axes[1].set_ylabel("Completeness (%)")
axes[1].set_ylim(0, 102)
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Normalize all series to base 100 at the first available date
norm = df_wide.div(df_wide.iloc[0]).mul(100)

final_perf = norm.iloc[-1].dropna().sort_values()
worst3     = final_perf.head(3).index.tolist()
best3      = final_perf.tail(3).index.tolist()
highlight  = set(worst3 + best3)

fig, ax = plt.subplots(figsize=(16, 7))

# All unremarkable tickers in gray
for col in norm.columns:
    if col not in highlight and col != BENCHMARK_TICKER:
        ax.plot(norm.index, norm[col], color="gray", alpha=0.4, linewidth=0.8)

# Worst 3 performers in red
for t in worst3:
    if t in norm.columns:
        ax.plot(norm.index, norm[t], color="red", linewidth=1.5, label=t)

# Best 3 performers in green
for t in best3:
    if t in norm.columns:
        ax.plot(norm.index, norm[t], color="green", linewidth=1.5, label=t)

# Benchmark in black
if BENCHMARK_TICKER in norm.columns:
    ax.plot(norm.index, norm[BENCHMARK_TICKER], color="black", linewidth=2, label=BENCHMARK_TICKER)

ax.set_title(f"Normalized price performance (base=100, {START_DATE})", fontweight="bold")
ax.set_ylabel("Indexed price")
ax.set_xlabel("")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Average daily volume per ticker (millions), sorted descending
avg_vol = (
    df_long.groupby("ticker")["volume"].mean()
    .sort_values(ascending=False)
    .div(1e6)
)

# Total market volume aggregated by date
total_vol      = df_long.groupby("date")["volume"].sum()
total_vol_roll = total_vol.rolling(30).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

# Top: average daily volume bar chart
axes[0].bar(avg_vol.index.tolist(), avg_vol.values, color="steelblue")
axes[0].set_title("Average daily volume by ticker", fontweight="bold")
axes[0].set_ylabel("Shares (millions)")
axes[0].set_xlabel("Ticker")
axes[0].tick_params(axis="x", rotation=45)

# Bottom: total market volume over time with 30-day rolling mean
axes[1].plot(total_vol.index, total_vol.div(1e6),
             color="lightgray", alpha=0.4, linewidth=0.8, label="Daily total")
axes[1].plot(total_vol_roll.index, total_vol_roll.div(1e6),
             color="steelblue", linewidth=1.5, label="30-day MA")
axes[1].set_title("Total market volume -- 30-day rolling mean", fontweight="bold")
axes[1].set_ylabel("Shares (millions)")
axes[1].set_xlabel("")
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 2 — Return distributions

In [ ]:
from scipy import stats

# Pooled returns across all tickers (flatten, drop NaN)
pooled = returns.values.flatten()
pooled = pooled[~np.isnan(pooled)]

# Per-ticker annualized stats
ann_ret = returns.mean() * 252 * 100
ann_vol = returns.std() * np.sqrt(252) * 100

ann_ret_sorted = ann_ret.sort_values(ascending=False)
ann_vol_sorted = ann_vol.sort_values(ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# -- Top-left: pooled return histogram -------------------------------------
mu, sigma = float(pooled.mean()), float(pooled.std())
skew_val  = float(stats.skew(pooled))
kurt_val  = float(stats.kurtosis(pooled))

sns.histplot(pooled, bins=100, kde=True, ax=axes[0, 0], color="steelblue")
axes[0, 0].axvline(mu,             color="blue", linestyle="--", linewidth=1.5,
                   label=f"Mean = {mu:.4f}")
axes[0, 0].axvline(mu + 2 * sigma, color="red",  linestyle="--", linewidth=1.2, label="+2 sigma")
axes[0, 0].axvline(mu - 2 * sigma, color="red",  linestyle="--", linewidth=1.2, label="-2 sigma")
txt_parts = [
    f"mu    = {mu:.4f}",
    f"sigma = {sigma:.4f}",
    f"skew  = {skew_val:.3f}",
    f"kurt  = {kurt_val:.3f}",
]
axes[0, 0].text(0.97, 0.95, chr(10).join(txt_parts),
                transform=axes[0, 0].transAxes,
                va="top", ha="right", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
axes[0, 0].set_title("Pooled daily log return distribution", fontweight="bold")
axes[0, 0].set_xlabel("Log return")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend(fontsize=8)

# -- Top-right: QQ-plot vs normal ------------------------------------------
(osm, osr), (slope, intercept, _r) = stats.probplot(pooled, dist="norm")
axes[0, 1].scatter(osm, osr, s=2, alpha=0.3, color="steelblue")
axes[0, 1].plot(osm, slope * np.array(osm) + intercept, color="red", linewidth=1.5)
axes[0, 1].set_title("QQ-plot vs normal", fontweight="bold")
axes[0, 1].set_xlabel("Theoretical quantiles")
axes[0, 1].set_ylabel("Sample quantiles")

# -- Bottom-left: annualized mean return per ticker ------------------------
colors_ret = ["green" if v >= 0 else "red" for v in ann_ret_sorted.values]
axes[1, 0].bar(ann_ret_sorted.index, ann_ret_sorted.values, color=colors_ret)
axes[1, 0].axhline(0, color="black", linewidth=0.8)
axes[1, 0].set_title("Annualized mean return (%)", fontweight="bold")
axes[1, 0].set_xlabel("Ticker")
axes[1, 0].set_ylabel("Return (%)")
axes[1, 0].tick_params(axis="x", rotation=45)

# -- Bottom-right: annualized volatility per ticker ------------------------
vol_colors = plt.cm.Oranges_r(np.linspace(0.2, 0.9, len(ann_vol_sorted)))
axes[1, 1].bar(ann_vol_sorted.index, ann_vol_sorted.values, color=vol_colors)
axes[1, 1].set_title("Annualized volatility (%)", fontweight="bold")
axes[1, 1].set_xlabel("Ticker")
axes[1, 1].set_ylabel("Volatility (%)")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Per-ticker single-day extremes
worst_day = returns.min().sort_values()
best_day  = returns.max().reindex(worst_day.index)

# SPY VaR and CVaR
spy_ret = returns[BENCHMARK_TICKER].dropna()
var_5   = float(np.percentile(spy_ret, 5))
var_1   = float(np.percentile(spy_ret, 1))
cvar_5  = float(spy_ret[spy_ret <= var_5].mean())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: horizontal grouped bar chart
y_pos = np.arange(len(worst_day))
axes[0].barh(y_pos - 0.2, worst_day.values, height=0.4, color="red",   label="Worst day")
axes[0].barh(y_pos + 0.2, best_day.values,  height=0.4, color="green", label="Best day")
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(worst_day.index.tolist(), fontsize=8)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Single-day return extremes by ticker", fontweight="bold")
axes[0].set_xlabel("Log return")
axes[0].legend()

# Right: SPY return distribution with VaR/CVaR annotations
sns.histplot(spy_ret, bins=80, kde=True, ax=axes[1], color="steelblue")
axes[1].axvline(var_5,  color="red",     linewidth=2,
                label=f"5% VaR  = {var_5:.4f}")
axes[1].axvline(var_1,  color="darkred", linewidth=2,
                label=f"1% VaR  = {var_1:.4f}")
axes[1].axvline(cvar_5, color="red",     linewidth=2, linestyle="--",
                label=f"5% CVaR = {cvar_5:.4f}")
axes[1].set_title("SPY daily return distribution -- VaR & CVaR", fontweight="bold")
axes[1].set_xlabel("Log return")
axes[1].set_ylabel("Count")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Section 3 — Correlation & rolling metrics

In [ ]:
corr_matrix = returns.corr()
spy_corr    = corr_matrix[BENCHMARK_TICKER].drop(BENCHMARK_TICKER).sort_values(ascending=False)
median_corr = float(spy_corr.median())

# Diverging colour palette for SPY-correlation bars
norm_vals  = (spy_corr - spy_corr.min()) / (spy_corr.max() - spy_corr.min())
bar_colors = [plt.cm.RdYlBu_r(float(v)) for v in norm_vals]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: full pairwise correlation heatmap
sns.heatmap(
    corr_matrix,
    ax=axes[0],
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    annot_kws={"size": 7},
)
axes[0].set_title("Pairwise return correlation (2015-2024)", fontweight="bold")
axes[0].tick_params(axis="x", rotation=45)
axes[0].tick_params(axis="y", rotation=0)

# Right: per-ticker correlation with SPY
axes[1].bar(spy_corr.index.tolist(), spy_corr.values, color=bar_colors)
axes[1].axhline(median_corr, color="black", linewidth=1.5, linestyle="--",
                label=f"Median = {median_corr:.3f}")
axes[1].set_title("Correlation with SPY", fontweight="bold")
axes[1].set_xlabel("Ticker")
axes[1].set_ylabel("Correlation")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Rolling window computations
roll_vol    = returns.rolling(60).std() * np.sqrt(252) * 100
roll_sharpe = (returns.rolling(60).mean() * 252) / (returns.rolling(60).std() * np.sqrt(252))

spy_series        = returns[BENCHMARK_TICKER]
equity_cols       = [c for c in returns.columns if c != BENCHMARK_TICKER]
roll_corr         = returns[equity_cols].apply(lambda col: col.rolling(60).corr(spy_series))
least_corr_ticker = roll_corr.mean().idxmin()

fig, axes = plt.subplots(3, 1, figsize=(16, 16), sharex=True)

def _shade(ax, start, end, color, label):
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=color, alpha=0.2, label=label)

# Row 1: 60-day rolling annualized volatility
for col in roll_vol.columns:
    if col != BENCHMARK_TICKER:
        axes[0].plot(roll_vol.index, roll_vol[col], color="gray", alpha=0.3, linewidth=0.8)
axes[0].plot(roll_vol.index, roll_vol[BENCHMARK_TICKER],
             color="black", linewidth=2, label=BENCHMARK_TICKER)
_shade(axes[0], "2020-01-01", "2020-12-31", "red",    "COVID-19")
_shade(axes[0], "2022-01-01", "2022-12-31", "orange", "Rate hikes")
axes[0].set_title("60-day rolling annualized volatility (%)", fontweight="bold")
axes[0].set_ylabel("Volatility (%)")
axes[0].legend(loc="upper right", fontsize=8)

# Row 2: SPY 60-day rolling Sharpe ratio
axes[1].plot(roll_sharpe.index, roll_sharpe[BENCHMARK_TICKER],
             color="steelblue", linewidth=1.5, label="SPY Sharpe")
axes[1].axhline(1, color="green", linestyle="--", linewidth=1.2, label="Sharpe = 1")
axes[1].axhline(0, color="gray",  linestyle="-",  linewidth=0.8)
axes[1].set_title("SPY 60-day rolling Sharpe ratio", fontweight="bold")
axes[1].set_ylabel("Sharpe ratio")
axes[1].legend(fontsize=8)

# Row 3: 60-day rolling correlation with SPY
for col in roll_corr.columns:
    if col != least_corr_ticker:
        axes[2].plot(roll_corr.index, roll_corr[col], color="gray", alpha=0.3, linewidth=0.8)
axes[2].plot(roll_corr.index, roll_corr[least_corr_ticker],
             color="steelblue", linewidth=2,
             label=f"{least_corr_ticker} (lowest avg corr)")
axes[2].set_title("60-day rolling correlation with SPY", fontweight="bold")
axes[2].set_ylabel("Correlation")
axes[2].legend(fontsize=8)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

In [ ]:
# Per-ticker stats for scatter
ann_ret_s = returns.mean() * 252 * 100
ann_vol_s = returns.std() * np.sqrt(252) * 100
sharpe_s  = (returns.mean() * 252) / (returns.std() * np.sqrt(252))
avg_vol_s = df_long.groupby("ticker")["volume"].mean()

common          = ann_ret_s.index.tolist()
avg_vol_aligned = avg_vol_s.reindex(common).fillna(avg_vol_s.mean())
vol_range       = avg_vol_aligned.max() - avg_vol_aligned.min()
if vol_range > 0:
    vol_norm = (avg_vol_aligned - avg_vol_aligned.min()) / vol_range
else:
    vol_norm = pd.Series(0.5, index=avg_vol_aligned.index)
bubble_sizes = vol_norm * 500 + 40

norm_c = plt.Normalize(float(sharpe_s.min()), float(sharpe_s.max()))
cmap   = plt.cm.RdYlGn

fig, ax = plt.subplots(figsize=(14, 8))

sc = ax.scatter(
    ann_vol_s[common].values,
    ann_ret_s[common].values,
    s=bubble_sizes[common].values,
    c=sharpe_s[common].values,
    cmap=cmap,
    norm=norm_c,
    alpha=0.8,
    zorder=3,
)

for ticker in common:
    if ticker != BENCHMARK_TICKER:
        ax.annotate(ticker,
                    xy=(ann_vol_s[ticker], ann_ret_s[ticker]),
                    xytext=(4, 4), textcoords="offset points",
                    fontsize=7.5)

if BENCHMARK_TICKER in common:
    ax.scatter(ann_vol_s[BENCHMARK_TICKER], ann_ret_s[BENCHMARK_TICKER],
               s=300, c="black", marker="*", zorder=5, label=BENCHMARK_TICKER)
    ax.annotate(BENCHMARK_TICKER,
                xy=(ann_vol_s[BENCHMARK_TICKER], ann_ret_s[BENCHMARK_TICKER]),
                xytext=(4, 4), textcoords="offset points",
                fontsize=8, fontweight="bold")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.axvline(ann_vol_s[common].mean(), color="gray", linewidth=0.8, linestyle="--",
           label="Market avg vol")

plt.colorbar(sc, ax=ax, label="Annualized Sharpe ratio")
ax.set_title("Risk-return profile (bubble size = avg daily volume)", fontweight="bold")
ax.set_xlabel("Annualized volatility (%)")
ax.set_ylabel("Annualized return (%)")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Section 4 — Key findings & strategy implications

In [ ]:
from scipy import stats as scipy_stats

# Recompute all summary stats from scratch so this cell is self-contained
ann_ret_f = returns.mean() * 252 * 100
ann_vol_f = returns.std() * np.sqrt(252) * 100
sharpe_f  = (returns.mean() * 252) / (returns.std() * np.sqrt(252))
pooled_f  = returns.values.flatten()
pooled_f  = pooled_f[~np.isnan(pooled_f)]

completeness_f   = df_wide.notna().sum() / len(df_wide) * 100
low_completeness = completeness_f[completeness_f < 99].to_dict()

equity_tickers = [t for t in returns.columns if t != BENCHMARK_TICKER]
corr_with_spy  = returns.corr()[BENCHMARK_TICKER].drop(BENCHMARK_TICKER)
pairwise_arr   = returns[equity_tickers].corr().values
np.fill_diagonal(pairwise_arr, np.nan)
avg_pair_corr  = float(np.nanmean(pairwise_arr))

sigma_f      = float(pooled_f.std())
fat_tail_pct = float((np.abs(pooled_f) > 3 * sigma_f).mean() * 100)

best_sharpe  = sharpe_f.idxmax()
worst_sharpe = sharpe_f.idxmin()
most_vol     = ann_vol_f.idxmax()
least_vol    = ann_vol_f.idxmin()
skew_f       = float(scipy_stats.skew(pooled_f))
kurt_f       = float(scipy_stats.kurtosis(pooled_f))
least_corr_t = corr_with_spy.idxmin()
most_corr_t  = corr_with_spy.idxmax()

SEP = "=" * 55

print(SEP)
print("DATA QUALITY")
print(SEP)
print(f"  Tickers loaded    : {returns.shape[1]} ({len(DEFAULT_TICKERS)} equity + benchmark)")
print(f"  Date range        : {df_wide.index.min().date()} to {df_wide.index.max().date()}")
if low_completeness:
    print(f"  Low completeness  : {low_completeness}")
else:
    print("  Low completeness  : none (all >= 99%)")

print()
print(SEP)
print("RETURN CHARACTERISTICS")
print(SEP)
print(f"  Highest Sharpe    : {best_sharpe} ({sharpe_f[best_sharpe]:.2f})")
print(f"  Lowest Sharpe     : {worst_sharpe} ({sharpe_f[worst_sharpe]:.2f})")
print(f"  Most volatile     : {most_vol} ({ann_vol_f[most_vol]:.1f}%)")
print(f"  Least volatile    : {least_vol} ({ann_vol_f[least_vol]:.1f}%)")
print(f"  Pooled skewness   : {skew_f:.3f}")
print(f"  Excess kurtosis   : {kurt_f:.3f}")
print(f"  |return| > 3s days: {fat_tail_pct:.2f}%")

print()
print(SEP)
print("CORRELATION STRUCTURE")
print(SEP)
print(f"  Avg pairwise corr : {avg_pair_corr:.3f}")
print(f"  Least corr w/ SPY : {least_corr_t} ({corr_with_spy[least_corr_t]:.3f})")
print(f"  Most corr w/ SPY  : {most_corr_t} ({corr_with_spy[most_corr_t]:.3f})")

print()
print(SEP)
print("STRATEGY IMPLICATIONS")
print(SEP)
print("  Momentum     : High cross-sectional return dispersion supports momentum strategy")
print("  Mean reversion: Fat tails and volatility clustering support short-term reversion")
print("  ML signal    : Low average pairwise correlation suggests diversifiable alpha")

## Notes

_Fill in manual observations after running the notebook:_

- Data quality issues found:
- Surprising return characteristics:
- Correlation anomalies:
- Strategy design implications: